# Vegetation Change (NDVI) Along the Swannanoa River Valley: Before vs. After Hurricane Helene

**Area of interest:** Swannanoa River corridor from Black Mountain, NC to its confluence with the French Broad River in Asheville (Biltmore Village) — one of the most severely flooded and wind-damaged stretches in Buncombe County during Hurricane Helene (landfall 2024-09-27).

**Method:** NDVI (Normalized Difference Vegetation Index) computed from NASA HLS (Harmonized Landsat Sentinel-2) 30 m surface reflectance, comparing a clear-sky scene shortly before Helene to one shortly after, to map negative NDVI change, a possible indicator of vegetation loss from wind damage, flooding, and landslides.

**Data source:** NASA LP DAAC, HLSL30/HLSS30 v2.0, accessed via `earthaccess` (NASA Earthdata Login required).

In [1]:
import sys
sys.path.append("..")

from pathlib import Path
from src.ndvi import compute_ndvi, ndvi_difference, align_to
from src.plotting import plot_before_after_diff, plot_change_histogram
from src.search_download import load_aoi_bbox

AOI_PATH = "../config/aoi_swannanoa.geojson"
aoi_bounds = load_aoi_bbox(AOI_PATH)
aoi_bounds

(np.float64(-82.58), np.float64(35.55), np.float64(-82.25), np.float64(35.65))

## 1. Acquire imagery

Run once from the project root (needs a free NASA Earthdata Login — https://urs.earthdata.nasa.gov/). The HLS `CLOUD_COVERAGE` attribute showed every scene in the narrow ±2-week windows around landfall was too cloudy (58%+), so the actual pre-Helene window was widened back into summer to find a usable match:

```bash
# Pre-Helene: clearest Landsat scene found between May and landfall (2024-09-27) -> 2024-08-30, 17% cloud
python src/search_download.py --start 2024-08-29 --end 2024-08-31 --max-cloud 20

# Post-Helene: clearest scene shortly after landfall -> 2024-10-09, 0% cloud
python src/search_download.py --start 2024-10-08 --end 2024-10-10 --max-cloud 20
```

Both are HLSL30 (Landsat 9), same MGRS tile (T17SLV), which avoids cross-sensor NDVI differences between Landsat and Sentinel-2. Downloaded `.tif` bands land in `data/raw/`.

In [2]:
RAW = Path("../data/raw")

# Pre-Helene: 2024-08-30, Landsat 9, 17% cloud cover (best available in summer 2024)
PRE = "HLS.L30.T17SLV.2024243T160534.v2.0"
PRE_RED = RAW / f"{PRE}.B04.tif"
PRE_NIR = RAW / f"{PRE}.B05.tif"
PRE_FMASK = RAW / f"{PRE}.Fmask.tif"

# Post-Helene: 2024-10-09, Landsat 9, 0% cloud cover — 12 days after landfall (2024-09-27)
POST = "HLS.L30.T17SLV.2024283T160541.v2.0"
POST_RED = RAW / f"{POST}.B04.tif"
POST_NIR = RAW / f"{POST}.B05.tif"
POST_FMASK = RAW / f"{POST}.Fmask.tif"

## 2. Compute NDVI (pre and post), with cloud/shadow masking

In [3]:
pre_ndvi = compute_ndvi(str(PRE_RED), str(PRE_NIR), str(PRE_FMASK), aoi_bounds=aoi_bounds)
post_ndvi = compute_ndvi(str(POST_RED), str(POST_NIR), str(POST_FMASK), aoi_bounds=aoi_bounds)

print("Pre-Helene mean NDVI: ", float(pre_ndvi.mean()))
print("Post-Helene mean NDVI:", float(post_ndvi.mean()))

Pre-Helene mean NDVI:  0.8095824718475342
Post-Helene mean NDVI: 0.7371037602424622


## 3. Difference map and NDVI change quantification

In [4]:
diff = ndvi_difference(pre_ndvi, post_ndvi)
post_aligned = align_to(pre_ndvi, post_ndvi)

fig = plot_before_after_diff(pre_ndvi, post_aligned, diff)
fig.savefig("../outputs/figures/ndvi_before_after_diff.png", dpi=200, bbox_inches="tight")

In [5]:
fig, pct_below = plot_change_histogram(diff, change_threshold=-0.15)
fig.savefig("../outputs/figures/ndvi_change_histogram.png", dpi=200, bbox_inches="tight")
print(f"{pct_below:.1f}% of valid pixels show ΔNDVI < -0.15 (substantial negative vegetation change)")

11.5% of valid pixels show ΔNDVI < -0.15 (substantial negative vegetation change)


## 4. (Stretch) Overlay flood extent or land cover change

Candidate supplementary datasets, if time allows:
- USGS/NOAA post-Helene flood extent polygons (high-water marks, flood inundation mapping)
- NC Emergency Management landslide inventory for Buncombe County
- NLCD land cover change product for broader context

These would be reprojected to match the NDVI grid and overlaid on the `ΔNDVI` raster to distinguish flood-driven riparian NDVI decline from wind-driven change upslope.